# 🗺️ Google Maps Platform API 종합 마스터 노트북 (서울 & 한국 중심)
### — 전체 입력 파라미터(Input) & 출력 속성(Output) & 서울 실무 비즈니스 시나리오 가이드

본 노트북은 **Google Maps Platform**의 모든 핵심 Web Service API 및 최신 **Places API (New / Modern v1)**의 모든 기능, 입력 옵션, 반환 속성을 **서울 주요 랜드마크 및 비즈니스 거점(강남, 여의도, 성수, 종로 등)**을 중심으로 검증하고 시각화할 수 있도록 구성되었습니다.

---

### 📌 다루는 API 서비스 및 서울 중심 실무 시나리오
| 번호 | API 서비스명 | 연동 방식 | 주요 입력(Inputs) & 출력(Outputs) | 서울 중심 실무 적용 시나리오 |
| :---: | :--- | :---: | :--- | :--- |
| **01** | **Geocoding API** | SDK / REST | 서울 주소, 컴포넌트 필터, Bounds ➡️ 좌표, 세부 행정구역 컴포넌트, 정밀도, 뷰포트 | 강남파이낸스센터(GFC) 도로명 주소 표준화, 한국 우편번호(06236) 한정 검색 |
| **02** | **Reverse Geocoding API** | SDK / REST | 위경도 좌표, ResultType, LocationType ➡️ 계층별 표준 주소 및 행정구역 매핑 | 강남역/역삼동 GPS 좌표를 "테헤란로" 도로명 주소로 변환, 배달/택시 픽업지 자동인식 |
| **03** | **Places API (New) - 검색** | REST v1 | TextQuery, Circle/Rect 반경, OpenNow, MinRating, Price, RankPreference ➡️ 장소 목록 | 강남역 주변 "영업중 + 평점 4.0 이상" 카페/맛집 검색, 성수동 핫플레이스 탐색 |
| **04** | **Places API (New) - 상세 (`*`)** | REST v1 | Place ID, FieldMask (`*`) ➡️ 50+ 속성 (리뷰, 영업시간, 편의시설, 전기차 충전, 휠체어) | 코엑스/더현대서울 매장 프로필 구축, 무장애(휠체어) 접근성 및 EV 충전소 정보 확인 |
| **05** | **Places Autocomplete API** | REST v1 | Partial Input, SessionToken, LocationBias, Origin ➡️ 추천 검색어, 직선거리, PlaceID | 서울시청 기준 "남산", "코엑스" 실시간 자동완성 및 거리(km) 계산, 세션 토큰 과금 절감 |
| **06** | **Directions API** | SDK / REST | Origin, Dest, Mode, Waypoints(`optimize:true`), TrafficModel, TransitOptions ➡️ 턴바이턴 경로, 소요시간 | 서울역 ➡️ 명동, 남산타워, 롯데월드몰 다중 경유지 최적 순서 배송(TSP), 실시간 올림픽대로/테헤란로 교통 반영 ETA |
| **07** | **Distance Matrix API** | SDK / REST | Origins[], Destinations[], Mode, DepartureTime, TrafficModel ➡️ N x M 거리 및 소요시간 | 3개 출발역(서울역, 용산역, 청량리역) ➡️ 2개 목적지(강남역, 여의도 더현대) 간 최적 배차 매트릭스 |
| **08** | **Elevation API** | SDK / REST | Locations[], Path[], Samples ➡️ 해발 고도, 측정 해상도, 고도 변화 프로파일 | 북한산 백운대, 남산타워, 롯데월드타워, 여의도 한강공원 고도 비교 및 한강변 경로 고도 프로파일 |
| **09** | **Time Zone API** | SDK / REST | Location, Timestamp ➡️ TimeZone ID, 표준 UTC 오프셋, 서머타임(DST) 오프셋 | 대한민국 서울(KST, UTC+9), 제주도, 독도 타임존 및 해외 주요 도시(뉴욕 DST) 비교 |
| **10** | **Geolocation API** | SDK / REST | ConsiderIP, CellTowers[], WiFiAccessPoints[] ➡️ 추정 좌표, 오차 반경 | 서울 시내 실내/지하철 등 GPS 음영지역 네트워크 기반 위치 추정 |
| **11** | **Roads API** | SDK / REST | Path[], Interpolate, Points[] ➡️ 도로 스냅 좌표, Place ID | 테헤란로 및 강남대로 주행 차량 GPS 궤적 도로 스냅 보정(Snap to Roads) |

---


## 📦 0. 환경 설정 및 패키지 설치 가이드

본 프로젝트는 초고속 패키지 관리자 `uv` 및 표준 `pip` 환경을 모두 지원합니다.

### ⚡ Option A. `uv` 사용 (권장)
```bash
# 가상환경 생성 및 의존성 설치
uv sync

# Jupyter 커널 등록
uv run python -m ipykernel install --user --name google_apis_env --display-name "Python (google_apis_env)"
```

### 🐍 Option B. 표준 `pip` / `venv` 사용
```bash
python3 -m venv .venv
source .venv/bin/activate
pip install googlemaps requests pandas python-dotenv ipykernel
python -m ipykernel install --user --name google_apis_env --display-name "Python (google_apis_env)"
```

### 🔑 `.env` 파일 설정
프로젝트 루트의 `.env` 파일에 Google Cloud Console에서 발급받은 API 키를 설정합니다:
```env
GOOGLE_MAPS_API_KEY=AIzaSy...your_actual_api_key_here
```


In [ ]:
import os
import json
import datetime
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
import googlemaps

# 루트 디렉토리의 .env 파일을 자동으로 탐색하고 로드
env_path = find_dotenv()
load_dotenv(env_path, override=True)

# 깔끔한 JSON 출력을 위한 헬퍼 함수
def print_json(data, title=None, max_lines=35):
    if title:
        print(f"\n=== {title} ===")
    formatted = json.dumps(data, indent=2, ensure_ascii=False, default=str)
    lines = formatted.splitlines()
    if len(lines) > max_lines:
        print("\n".join(lines[:max_lines]))
        print(f"... [총 {len(lines)}줄 중 {max_lines}줄 출력됨 - 전체 데이터는 반환 객체 변수 참조]")
    else:
        print(formatted)

print(f"✅ 환경 설정 및 .env 로드 완료: {env_path if env_path else '기본 환경변수 사용'}")


## 🔑 1. API 키 로드 및 클라이언트 초기화


In [ ]:
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

if API_KEY:
    API_KEY = API_KEY.strip().strip('"').strip("'")

if not API_KEY or API_KEY == "YOUR_GOOGLE_MAPS_API_KEY_HERE":
    import getpass
    API_KEY = getpass.getpass("Google Maps API Key를 입력하세요: ").strip().strip('"').strip("'")

try:
    gmaps = googlemaps.Client(key=API_KEY)
    masked_key = f"{API_KEY[:6]}...{API_KEY[-4:]}" if len(API_KEY) > 10 else "***"
    print(f"✅ Google Maps Python SDK 클라이언트 초기화 성공 (키: {masked_key})")
except Exception as e:
    print("❌ 클라이언트 초기화 실패:", e)


## 📍 2. Geocoding API (서울 주소 ➡️ 좌표 지오코딩)

### 📥 지원 입력 파라미터 (Input Attributes):
- `address`: 검색할 주소 문자열 (예: `"서울특별시 강남구 테헤란로 152 강남파이낸스센터"`)
- `components`: 특정 국가/우편번호/행정구역으로 강제 필터링 (예: `{'country': 'KR', 'postal_code': '06236'}`)
- `bounds`: 검색 바운딩 박스 가중치 (서울 영역 남서/북동 좌표 튜플)
- `region`: 국가 코드(ccTLD) 바이어스 (`"kr"`)
- `language`: 응답 주소 표기 언어 (`"ko"`)

### 📤 반환 출력 속성 (Output Attributes):
- `formatted_address`: 표준 도로명 전체 주소
- `place_id`: Google 고유 장소 식별자
- `types`: 주소 유형 (`street_address`, `premise`, `sublocality_level_1` 등)
- `geometry`:
  - `location`: 위도(`lat`), 경도(`lng`)
  - `location_type`: 위치 정밀도 (`ROOFTOP`, `RANGE_INTERPOLATED`, `GEOMETRIC_CENTER`, `APPROXIMATE`)
  - `viewport` / `bounds`: 서울 지도 뷰포트 표시 영역
- `address_components`: 도로명, 건물번호, 동/구/시/도, 국가, 우편번호별 상세 분해 객체 배열

### 💡 실무 적용 시나리오:
1. **국내 커머스/물류 배송 주소 정제**: 고객이 입력한 불완전한 도로명 주소를 표준 도로명 주소 체계로 변환 및 우편번호(5자리 기초구역번호) 검증
2. **한국 영역 한정 검색 (Component Restriction)**: 해외 주소 오매칭 방지를 위해 대한민국(`country:KR`) 및 서울 지역으로 한정
3. **지도 줌 바운딩박스 자동 계산**: 반환된 `viewport` 영역을 통해 강남파이낸스센터(GFC) 적정 지도 줌 레벨 자동 설정


In [ ]:
# 시나리오 1: 서울 강남파이낸스센터(GFC) 표준 지오코딩 및 행정구역 컴포넌트 분석
seoul_target_address = "서울특별시 강남구 테헤란로 152 강남파이낸스센터"

try:
    geocode_result = gmaps.geocode(seoul_target_address, language="ko", region="kr")
    print(f"✅ [시나리오 1] 서울 주소 '{seoul_target_address}' 조회 결과 {len(geocode_result)}건:")
    
    if geocode_result:
        first = geocode_result[0]
        print(f"  - 표준 주소: {first.get('formatted_address')}")
        print(f"  - Place ID: {first.get('place_id')}")
        print(f"  - 위치 정밀도: {first.get('geometry', {}).get('location_type')}")
        loc = first.get('geometry', {}).get('location', {})
        print(f"  - 위도/경도: lat={loc.get('lat')}, lng={loc.get('lng')}")
        
        # 주소 컴포넌트 테이블
        df_comp = pd.DataFrame(first.get('address_components', []))
        display(df_comp)

    # 시나리오 2: 한국 우편번호(06236) 컴포넌트 필터링을 적용한 정밀 검색
    filtered_result = gmaps.geocode(
        "테헤란로 152",
        components={"country": "KR", "postal_code": "06236"},
        language="ko"
    )
    if filtered_result:
        print(f"\n✅ [시나리오 2] 한국 컴포넌트 필터 적용 검색 (대한민국 06236 테헤란로 152):")
        print(f"  - 표준 주소: {filtered_result[0].get('formatted_address')}")
        print(f"  - 좌표: {filtered_result[0].get('geometry', {}).get('location')}")
        
except Exception as e:
    print("❌ Geocoding API 오류:", e)


## 🔄 3. Reverse Geocoding API (서울 좌표 ➡️ 주소 역지오코딩)

### 📥 지원 입력 파라미터 (Input Attributes):
- `latlng`: 서울 좌표 튜플 (예: 강남구 테헤란로 `(37.50005, 127.0365)` 또는 서울시청 `(37.5665, 126.9780)`)
- `result_type`: 특정 주소 유형으로 결과 필터링 (예: `['street_address']`, `['premise']`, `['political']`)
- `location_type`: 위치 정밀도로 필터링 (예: `['ROOFTOP']`, `['APPROXIMATE']`)
- `language`: 결과 언어 (`"ko"`)

### 📤 반환 출력 속성 (Output Attributes):
- 정밀한 건물/지번 단위 도로명 주소부터 동(역삼동)/구(강남구)/시(서울특별시)/국가(대한민국) 단위까지 계층화된 주소 매칭 후보 목록

### 💡 실무 적용 시나리오:
1. **서울 라이드헤일링/배달 앱 픽업 위치 자동 인식**: 사용자의 현재 GPS 좌표를 실시간 "서울특별시 강남구 역삼동 테헤란로 152" 도로명 주소로 표시
2. **ROOFTOP 필터 기반 정밀 승하차 지점 추출**: `location_type=['ROOFTOP']`로 오차 없이 실제 건물 출입구 도로명 주소 특정
3. **서울 자치구별 행정구역 자동 분기**: 강남구, 중구, 성동구 등 행정구역 코드를 파싱하여 지역 기반 맞춤 프로모션 제공


In [ ]:
# 서울 역삼동 강남파이낸스센터(GFC) 인근 좌표
seoul_coords = (37.50005, 127.0365)

try:
    # 1. 일반 역지오코딩 (서울시 행정 계층별 주소 전체 반환)
    rev_all = gmaps.reverse_geocode(seoul_coords, language="ko")
    print(f"✅ 서울 좌표 {seoul_coords} 일반 역지오코딩: 총 {len(rev_all)}개 계층 결과 반환\n")
    
    rev_records = []
    for idx, item in enumerate(rev_all[:6]):
        rev_records.append({
            "순번": idx + 1,
            "표준 주소": item.get("formatted_address"),
            "정밀도 (Location Type)": item.get("geometry", {}).get("location_type"),
            "주소 유형 (Types)": ", ".join(item.get("types", []))
        })
    df_rev = pd.DataFrame(rev_records)
    display(df_rev)

    # 2. 정밀 필터링: ROOFTOP 정밀도의 건물/도로명 주소만 추출
    rev_filtered = gmaps.reverse_geocode(
        seoul_coords,
        result_type=["street_address", "premise"],
        location_type=["ROOFTOP"],
        language="ko"
    )
    if rev_filtered:
        print("🎯 [ROOFTOP 정밀 필터 결과]:", rev_filtered[0].get("formatted_address"))
except Exception as e:
    print("❌ Reverse Geocoding API 오류:", e)


## 🔍 4. Places API (New): 서울 장소 텍스트 검색 & 주변 시설 검색

최신 **Places API Modern v1** 엔드포인트를 사용하여 **서울 강남/성수/명동 등 주요 핫플레이스**의 장소를 탐색합니다.

### 📥 지원 입력 파라미터 (Input Attributes):
- `textQuery`: 검색어 (예: `"강남파이낸스센터"`, `"성수동 카페"`)
- `includedType` / `includedTypes`: 장소 카테고리 (예: `["cafe", "korean_restaurant", "bakery", "hotel"]`)
- `locationRestriction` / `locationBias`: 서울 중심 검색 반경 (중심점 `center` + 반경 `radius` 미터)
- `openNow`: 현재 영업 중인 서울 매장만 필터링 (`true`/`false`)
- `minRating`: 최소 평점 필터 (예: `4.0`, `4.5`)
- `priceLevels`: 가격대 필터
- `rankPreference`: 정렬 기준 (`POPULARITY`(인기순), `DISTANCE`(거리순))
- `maxResultCount`: 결과 개수 (1~20)
- `X-Goog-FieldMask`: 반환받을 필드 목록

### 📤 반환 출력 속성 (Output Attributes):
- 장소 ID, 표시 상호명, 서울 표준 주소, 평점, 총 리뷰수, 영업상태, 대표 카테고리, 좌표, 영업시간 등

### 💡 실무 적용 시나리오:
1. **조건부 서울 맛집/카페 큐레이션**: "강남역 인근 평점 4.0 이상이며 현재 영업 중인 카페" 필터링
2. **거리순 긴급 편의시설 탐색**: 서울시청/강남역 반경 1.5km 내 가장 가까운 편의점/약국/주차장 거리순 정렬


In [ ]:
# 4.1 Places Text Search (New v1): 서울 강남파이낸스센터 및 주요 랜드마크 검색
url_text = "https://places.googleapis.com/v1/places:searchText"
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.userRatingCount,places.location,places.primaryType,places.businessStatus,places.regularOpeningHours"
}

body_text = {
    "textQuery": "강남파이낸스센터",
    "minRating": 4.0,
    "languageCode": "ko",
    "regionCode": "kr"
}

sample_seoul_place_id = None
res_text = requests.post(url_text, headers=headers, json=body_text)

if res_text.status_code == 200:
    data = res_text.json()
    places = data.get("places", [])
    print(f"✅ 서울 Places 텍스트 검색 성공 ({len(places)}건 반환):")
    
    rows = []
    for p in places:
        rows.append({
            "장소명": p.get("displayName", {}).get("text"),
            "평점": p.get("rating"),
            "리뷰 수": p.get("userRatingCount"),
            "카테고리": p.get("primaryType"),
            "영업 상태": p.get("businessStatus"),
            "주소": p.get("formattedAddress"),
            "Place ID": p.get("id")
        })
    df_text = pd.DataFrame(rows)
    display(df_text)
    if places:
        sample_seoul_place_id = places[0].get("id")
        print(f"🎯 상세 조회용 서울 Place ID 선택: {sample_seoul_place_id}")
else:
    print(f"❌ Text Search 오류 ({res_text.status_code}):", res_text.text)

# 4.2 Places Nearby Search (New v1): 강남역/역삼동 반경 1,500m 내 카페 검색
url_nearby = "https://places.googleapis.com/v1/places:searchNearby"
headers_nearby = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.userRatingCount,places.primaryType,places.location"
}

body_nearby = {
    "includedTypes": ["cafe", "coffee_shop", "bakery"],
    "maxResultCount": 5,
    "locationRestriction": {
        "circle": {
            "center": {"latitude": 37.50005, "longitude": 127.0365},  # 강남 테헤란로 중심
            "radius": 1500.0  # 반경 1,500 미터
        }
    },
    "languageCode": "ko"
}

res_nearby = requests.post(url_nearby, headers=headers_nearby, json=body_nearby)
if res_nearby.status_code == 200:
    nearby_data = res_nearby.json()
    nearby_places = nearby_data.get("places", [])
    print(f"\n✅ 서울 강남 테헤란로 반경 1.5km 검색 성공 ({len(nearby_places)}개 매장 발견):")
    df_nearby = pd.DataFrame([
        {
            "장소명": p.get("displayName", {}).get("text"),
            "평점": p.get("rating"),
            "리뷰 수": p.get("userRatingCount"),
            "주소": p.get("formattedAddress"),
            "위치": p.get("location")
        } for p in nearby_places
    ])
    display(df_nearby)
else:
    print(f"❌ Nearby Search 오류 ({res_nearby.status_code}):", res_nearby.text)


## 🏢 5. Places API (New): 서울 장소 상세 정보 전체 속성 조회 (`*` Wildcard FieldMask)

와일드카드 필드마스크 `X-Goog-FieldMask: *`를 지정하여 서울 주요 시설(GFC, 코엑스 등)에 대해 Google이 보유한 **모든 속성(50여 개 이상의 필드)**을 완전히 추출합니다.

### 📥 지원 입력 파라미터 (Input Attributes):
- `place_id`: 서울 장소 고유 식별자 (예: 강남파이낸스센터 Place ID)
- `X-Goog-FieldMask`: `*` (전체 조회)
- `languageCode`: `"ko"`

### 📤 제공되는 전체 속성 카테고리 (Output Attributes):
1. **식별 및 지오메트리**: `id`, `displayName`, `formattedAddress`, `location`, `viewport`, `plusCode`, `googleMapsUri`, `types`, `primaryType`
2. **연락처 및 웹**: `internationalPhoneNumber`, `nationalPhoneNumber`, `websiteUri`
3. **영업 시간**: `regularOpeningHours`, `currentOpeningHours`, `regularSecondaryOpeningHours`
4. **소개 및 리뷰**: `editorialSummary`, `rating`, `userRatingCount`, `priceLevel`, `reviews` (작성자, 평점, 본문, 작성일, 원문)
5. **다이닝 & 서비스 옵션**: `dineIn`, `delivery`, `takeout`, `curbsidePickup`, `reservable`, `servesBreakfast`, `servesLunch`, `servesDinner`, `servesBeer`, `servesWine`, `servesVegetarianFood`
6. **현대 시설 & 분위기**: `outdoorSeating`, `liveMusic`, `menuForChildren`, `goodForChildren`, `goodForGroups`, `restroom`, `allowsDogs`
7. **접근성 (Accessibility)**: `wheelchairAccessibleParking`, `wheelchairAccessibleEntrance`, `wheelchairAccessibleRestroom`, `wheelchairAccessibleSeating`
8. **주차 및 전기차(EV)**: `parkingOptions` (무료/유료 주차장, 발렛 등), `evChargeOptions` (충전 포트 수 및 커넥터 타입)
9. **결제 수단**: `paymentOptions` (신용카드, 체크카드, 현금전용, NFC 간편결제)
10. **사진(Photos)**: 사진 레퍼런스 및 해상도, 저작권 attribution

### 💡 실무 적용 시나리오:
1. **서울 매장 상세 프로필 카드 렌더링**: 영업시간, 연락처, 고객 리뷰, 대표 사진을 단 1회의 호출로 표기
2. **서울 무장애(배리어프리) 관광 지도**: 휠체어 전용 주차장/출입구/화장실 완비 여부 필터링
3. **서울 시내 EV 전기차 충전소 및 반려동물 동반 가능 매장 안내**


In [ ]:
# 서울 강남파이낸스센터(GFC) 또는 검색된 장소의 전체 상세 속성 조회
target_place_id = sample_seoul_place_id or "ChIJj61dQgK6j4AR4GeTYWZsKWw"

url_details = f"https://places.googleapis.com/v1/places/{target_place_id}"
headers_details = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "*"  # 모든 필드 일괄 요청
}

res_details = requests.get(url_details, headers=headers_details)

if res_details.status_code == 200:
    place_all = res_details.json()
    print(f"✅ 서울 장소 상세 전체 페이로드 수신 성공! (제공된 최상위 필드 수: {len(place_all.keys())}개)\n")
    
    # 1. 기본 식별 및 연락처
    print("📌 [1. 기본 식별 및 연락처]")
    print(f"  - 장소명: {place_all.get('displayName', {}).get('text')}")
    print(f"  - Place ID: {place_all.get('id')}")
    print(f"  - 표준 주소: {place_all.get('formattedAddress')}")
    print(f"  - 전화번호: {place_all.get('internationalPhoneNumber', '제공안됨')}")
    print(f"  - 웹사이트: {place_all.get('websiteUri', '제공안됨')}")
    print(f"  - 소개 요약: {place_all.get('editorialSummary', {}).get('text', '정보 없음')}")
    
    # 2. 시설 편의성 및 옵션 분해
    print("\n🍽️ [2. 편의시설, 접근성 및 서비스 옵션]")
    feature_dict = {
        "매장 내 식사 (Dine In)": place_all.get("dineIn"),
        "배달 (Delivery)": place_all.get("delivery"),
        "포장 (Takeout)": place_all.get("takeout"),
        "예약 가능 (Reservable)": place_all.get("reservable"),
        "채식 메뉴 (Vegetarian)": place_all.get("servesVegetarianFood"),
        "야외 좌석 (Outdoor Seating)": place_all.get("outdoorSeating"),
        "반려동물 허용 (Allows Dogs)": place_all.get("allowsDogs"),
        "휠체어 접근성": place_all.get("accessibilityOptions"),
        "주차 옵션": place_all.get("parkingOptions"),
        "결제 수단": place_all.get("paymentOptions"),
        "전기차 충전 (EV)": place_all.get("evChargeOptions")
    }
    for k, v in feature_dict.items():
        if v is not None:
            print(f"  - {k}: {v}")

    # 3. 고객 리뷰 테이블
    reviews = place_all.get("reviews", [])
    print(f"\n💬 [3. 고객 리뷰 내역 ({len(reviews)}건)]")
    review_rows = []
    for r in reviews:
        review_rows.append({
            "작성자": r.get("authorAttribution", {}).get("displayName"),
            "평점": f"⭐ {r.get('rating')}",
            "작성일": r.get("relativePublishTimeDescription"),
            "리뷰": (r.get("text", {}).get("text", "")[:80] + "...") if len(r.get("text", {}).get("text", "")) > 80 else r.get("text", {}).get("text", "")
        })
    df_reviews = pd.DataFrame(review_rows)
    display(df_reviews)
    
    # 4. 전체 JSON 출력 (상위 30줄)
    print_json(place_all, title="Place Details 전체 원본 응답 (*)")
else:
    print(f"❌ Place Details 오류 ({res_details.status_code}):", res_details.text)


## ✍️ 6. Places Autocomplete API (서울 랜드마크 실시간 자동완성 & 세션 토큰)

### 📥 지원 입력 파라미터 (Input Attributes):
- `input`: 사용자 실시간 입력 문자열 (예: `"남산"`, `"코엑스"`, `"경복궁"`)
- `sessionToken`: 자동완성 타이핑 ➡️ 최종 장소 선택까지 묶어 과금을 최적화하는 UUID 토큰
- `origin`: 기준 좌표 (서울시청 `(37.5665, 126.9780)`을 기준으로 후보 장소까지의 **직선거리(distanceMeters)** 자동 계산)
- `includedRegionCodes`: 추천 후보 국가 제한 (`["kr"]`)
- `includedPrimaryTypes`: 특정 장소 유형만 추천 (예: `["tourist_attraction", "park", "establishment"]`)

### 📤 반환 출력 속성 (Output Attributes):
- `suggestions`:
  - `placePrediction`:
    - `placeId`: 서울 장소 ID
    - `text`: 전체 추천 텍스트
    - `structuredFormat`: 메인 장소명(`mainText`) 및 보조 주소(`secondaryText`) 분리 구조체
    - `distanceMeters`: 서울시청으로부터의 직선거리(미터)
    - `types`: 장소 유형 태그

### 💡 실무 적용 시나리오:
1. **서울 포털/앱 검색창 인스턴트 자동완성**: 키보드 입력 시 "남산서울타워", "남산골한옥마을" 등을 분리 표시하고 사용자 위치로부터의 거리(km) 실시간 렌더링
2. **Session Token 기반 API 과금 최적화**: 사용자의 10회 타이핑 호출을 1회의 세션으로 묶어 비용 절감


In [ ]:
import uuid

# 세션 토큰 생성 (UUID v4)
session_token = str(uuid.uuid4())

url_auto = "https://places.googleapis.com/v1/places:autocomplete"
headers_auto = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY
}

# 서울시청 기준 "남산" 자동완성 검색
body_auto = {
    "input": "남산",
    "sessionToken": session_token,
    "origin": {
        "latitude": 37.5665,
        "longitude": 126.9780  # 서울시청 기준점
    },
    "includedRegionCodes": ["kr"],
    "languageCode": "ko"
}

res_auto = requests.post(url_auto, headers=headers_auto, json=body_auto)

if res_auto.status_code == 200:
    auto_data = res_auto.json()
    suggestions = auto_data.get("suggestions", [])
    print(f"✅ 서울 자동완성 결과 {len(suggestions)}건 반환 (서울시청 기준 거리 계산):\n")
    
    sugg_list = []
    for s in suggestions:
        pred = s.get("placePrediction", {})
        dist_m = pred.get("distanceMeters")
        dist_km_str = f"{dist_m / 1000:.2f} km" if dist_m is not None else "계산불가"
        
        sugg_list.append({
            "메인 명칭": pred.get("structuredFormat", {}).get("mainText", {}).get("text"),
            "상세 주소": pred.get("structuredFormat", {}).get("secondaryText", {}).get("text"),
            "서울시청 기준 거리": dist_km_str,
            "Place ID": pred.get("placeId"),
            "유형": ", ".join(pred.get("types", [])[:3])
        })
    df_sugg = pd.DataFrame(sugg_list)
    display(df_sugg)
else:
    print(f"❌ Autocomplete 오류 ({res_auto.status_code}):", res_auto.text)


## 🚗 7. Directions API (서울 도심 경로 탐색 & 다중 경유지 TSP 최적화)

### 📥 지원 입력 파라미터 (Input Attributes):
- `origin`: 출발지 (예: `"서울역"`)
- `destination`: 목적지 (예: `"코엑스"`)
- `waypoints`: 서울 주요 경유지 목록 (`["명동성당", "남산서울타워", "잠실 롯데월드몰"]`)
- `optimize_waypoints`: `True` 설정 시 외판원 문제(TSP)를 해결하여 **서울 시내 최단 주행 순서로 경유지 자동 재배열**
- `mode`: 이동 수단 (`"driving"`, `"transit"`, `"walking"`)
- `departure_time`: `"now"` (올림픽대로, 강남대로, 한강대교 실시간 교통 상황 반영)
- `traffic_model`: `"best_guess"`, `"pessimistic"`, `"optimistic"`
- `avoid`: `["tolls"]`(남산 1/3호 터널 혼잡통행료 등 유료도로 회피), `["highways"]`

### 📤 반환 출력 속성 (Output Attributes):
- `routes`:
  - `summary`: 주요 경유 도로명 (예: 한강대로, 올림픽대로, 테헤란로)
  - `waypoint_order`: 최적화된 서울 경유지 방문 순서 인덱스 배열
  - `legs`: 각 구간별 거리(`distance`), 기본 소요시간(`duration`), 실시간 교통소요시간(`duration_in_traffic`)
  - `steps`: 턴바이턴 회전 안내(HTML 및 클린 텍스트), 동작(`maneuver`)
  - `overview_polyline`: 지도에 표시할 인코딩 경로 폴리라인

### 💡 실무 적용 시나리오:
1. **서울 당일배송/퀵서비스 다중 배송지 순서 최적화 (`optimize_waypoints=True`)**: 서울역에서 출발하여 명동, 남산, 잠실, 코엑스를 최단 시간으로 순회하는 최적 동선 도출
2. **서울 출퇴근 시간대 정체 예측**: 비오는 날/출근 시간대 `traffic_model="pessimistic"`을 적용하여 여유로운 배송 도착 시간 산출
3. **도심 내비게이션 턴바이턴 가이드**: 교차로 우회전/좌회전 및 고가도로 진입 안내


In [ ]:
# 서울역 출발 -> 코엑스 도착 (명동, 남산, 잠실 경유지 최적 순서 정렬)
origin = "서울특별시 중구 한강대로 405 서울역"
destination = "서울특별시 강남구 영동대로 513 코엑스"
waypoints = [
    "서울특별시 중구 명동길 74 명동대성당",
    "서울특별시 용산구 남산공원길 105 N서울타워",
    "서울특별시 송파구 올림픽로 300 롯데월드몰"
]

try:
    # optimize_waypoints=True 설정으로 최적 방문 순환 코스 계산
    seoul_directions = gmaps.directions(
        origin=origin,
        destination=destination,
        waypoints=waypoints,
        optimize_waypoints=True,
        mode="driving",
        departure_time="now",
        traffic_model="best_guess"
    )

    if seoul_directions:
        route = seoul_directions[0]
        print(f"✅ 서울 다중 경유지 경로 최적화 완료 (주요 도로: {route.get('summary')}):")
        print(f"🎯 최적화된 경유지 방문 순서 인덱스: {route.get('waypoint_order')}")
        
        # 각 구간(Leg)별 거리 및 소요시간 요약
        leg_rows = []
        for i, leg in enumerate(route.get("legs", [])):
            leg_rows.append({
                "구간": f"구간 {i + 1}",
                "출발지": leg.get("start_address")[:25] + "...",
                "도착지": leg.get("end_address")[:25] + "...",
                "구간 거리": leg.get("distance", {}).get("text"),
                "기본 소요시간": leg.get("duration", {}).get("text"),
                "실시간 소요시간": leg.get("duration_in_traffic", {}).get("text", "N/A")
            })
        df_legs = pd.DataFrame(leg_rows)
        display(df_legs)

        # 첫 번째 구간의 상위 5개 턴바이턴 주행 안내
        steps = route["legs"][0]["steps"]
        print(f"\n📋 [서울역 출발 첫 번째 구간의 단계별 턴바이턴 가이드 (상위 5단계)]:")
        import re
        step_rows = []
        for s_idx, s in enumerate(steps[:5]):
            step_rows.append({
                "단계": s_idx + 1,
                "주행 안내": re.sub('<[^<]+?>', '', s.get("html_instructions", "")),
                "거리": s.get("distance", {}).get("text"),
                "시간": s.get("duration", {}).get("text"),
                "동작": s.get("maneuver", "직진")
            })
        display(pd.DataFrame(step_rows))

except Exception as e:
    print("❌ Directions API 오류:", e)


## 📏 8. Distance Matrix API (서울 주요 거점 간 N x M 거리 행렬)

### 📥 지원 입력 파라미터 (Input Attributes):
- `origins`: 서울 주요 출발지 목록 (예: 서울 주요 철도역 `["서울역", "용산역", "청량리역"]`)
- `destinations`: 서울 주요 목적지 목록 (예: 핵심 상권 `["강남역", "여의도 더현대 서울"]`)
- `mode`: 이동 수단 (`"driving"`, `"transit"`, `"walking"`)
- `departure_time`: `"now"` (실시간 서울 교통량 반영)
- `traffic_model`: `"best_guess"`

### 📤 반환 출력 속성 (Output Attributes):
- `origin_addresses` / `destination_addresses`: 정규화된 서울 주소 배열
- `rows[i].elements[j]`: $i$번째 출발지와 $j$번째 목적지 간의:
  - `distance`: 미터(`value`) 및 문자열(`text`)
  - `duration`: 초(`value`) 및 문자열(`text`)
  - `duration_in_traffic`: 서울 실시간 교통 반영 소요시간

### 💡 실무 적용 시나리오:
1. **서울 라이드헤일링/배달 기사 최적 매칭**: 3개 거점의 대기 기사와 2개 목적지의 승객 간 N x M 소요시간을 계산하여 최단 시간 도착 기사 자동 배차
2. **서울 물류 거점 센터 후보지 분석**: 서울역/용산역/청량리역 중 강남 및 여의도로의 평균 배송 이동 비용/시간이 가장 적은 최적 거점 도출


In [ ]:
seoul_origins = [
    "서울특별시 중구 한강대로 405 서울역",
    "서울특별시 용산구 한강대로23길 55 용산역",
    "서울특별시 동대문구 왕산로 214 청량리역"
]

seoul_destinations = [
    "서울특별시 강남구 강남대로 396 강남역",
    "서울특별시 영등포구 여의대로 108 더현대 서울"
]

try:
    matrix_result = gmaps.distance_matrix(
        origins=seoul_origins,
        destinations=seoul_destinations,
        mode="driving",
        departure_time="now",
        traffic_model="best_guess"
    )

    matrix_rows = []
    for i, origin_name in enumerate(matrix_result.get("origin_addresses", [])):
        row_elements = matrix_result["rows"][i]["elements"]
        for j, dest_name in enumerate(matrix_result.get("destination_addresses", [])):
            element = row_elements[j]
            if element.get("status") == "OK":
                matrix_rows.append({
                    "출발지 (기사/물류거점)": origin_name,
                    "목적지 (고객/도착점)": dest_name,
                    "거리": element.get("distance", {}).get("text"),
                    "표준 소요시간": element.get("duration", {}).get("text"),
                    "실시간 소요시간 (교통반영)": element.get("duration_in_traffic", {}).get("text", "정보 없음")
                })

    df_matrix = pd.DataFrame(matrix_rows)
    print(f"✅ 서울 {len(seoul_origins)}개 출발지 x {len(seoul_destinations)}개 목적지 거리 행렬 계산 완료:")
    display(df_matrix)
except Exception as e:
    print("⚠️ Distance Matrix API 참고:", e)
    print("💡 콘솔 활성화 링크: https://console.cloud.google.com/apis/library/distancematrix-backend.googleapis.com")


## ⛰️ 9. Elevation API (서울 주요 지형 고도 측정 & 남산 경로 프로파일)

### 📥 지원 입력 파라미터 (Input Attributes):
- `locations`: 서울 대표 랜드마크 좌표 목록 (북한산, 남산타워, 롯데월드타워, 여의도 한강공원)
- `path` & `samples`: 서울역에서 남산서울타워 정상까지의 경로 좌표 목록과 샘플링 포인트 수

### 📤 반환 출력 속성 (Output Attributes):
- `elevation`: 해발 고도 (미터 단위 부동소수점)
- `location`: 위도/경도
- `resolution`: 고도 데이터 측정 해상도 (미터 단위)

### 💡 실무 적용 시나리오:
1. **서울 등산/트레킹 코스 경사도 및 획득고도 분석**: 서울역 ➡️ 남산타워 코스의 오르막 고도 프로파일 생성
2. **도심 드론/UAV 도심 항공교통(UAM) 지형 안전고도 확보**: 한강 회랑 및 도심 비행 구간의 지형 최고 고도 데이터 검증


In [ ]:
# 1. 서울 대표 랜드마크 고도 비교
seoul_landmarks = [
    {"name": "북한산 백운대 (서울 최고봉 836m)", "coords": (37.6587, 126.9781)},
    {"name": "남산서울타워 (남산 정상 243m)", "coords": (37.5512, 126.9882)},
    {"name": "잠실 롯데월드타워 (지표면 고도)", "coords": (37.5126, 127.1026)},
    {"name": "여의도 한강공원 (수변 해발고도)", "coords": (37.5283, 126.9246)}
]

try:
    coords_list = [l["coords"] for l in seoul_landmarks]
    elevation_results = gmaps.elevation(coords_list)
    
    elevation_records = []
    for l, res in zip(seoul_landmarks, elevation_results):
        elev_m = res.get("elevation", 0)
        elevation_records.append({
            "위치 명칭": l["name"],
            "위도": res.get("location", {}).get("lat"),
            "경도": res.get("location", {}).get("lng"),
            "해발 고도 (미터)": f"{elev_m:.2f} m",
            "해발 고도 (피트)": f"{elev_m * 3.28084:.2f} ft",
            "측정 해상도": f"{res.get('resolution', 0):.2f} m"
        })
    df_elevation = pd.DataFrame(elevation_records)
    print("✅ 서울 주요 랜드마크 해발 고도 측정 결과:")
    display(df_elevation)

    # 2. 서울역 -> 남산서울타워 경로 고도 프로파일 샘플링 (5개 지점)
    seoul_path_sample = [(37.5563, 126.9723), (37.5512, 126.9882)]
    path_elev = gmaps.elevation_along_path(seoul_path_sample, samples=5)
    print(f"\n✅ 서울역 ➡️ 남산타워 오르막 경로 고도 프로파일 ({len(path_elev)}개 지점):")
    df_path = pd.DataFrame([
        {
            "샘플 지점": f"Point #{idx + 1}",
            "좌표": f"({p['location']['lat']:.4f}, {p['location']['lng']:.4f})",
            "고도 (m)": f"{p['elevation']:.2f} m"
        } for idx, p in enumerate(path_elev)
    ])
    display(df_path)

except Exception as e:
    print("⚠️ Elevation API 참고:", e)
    print("💡 콘솔 활성화 링크: https://console.cloud.google.com/apis/library/elevation-backend.googleapis.com")


## ⏰ 10. Time Zone API (서울 KST 표준시 & 글로벌 타임존 비교)

### 📥 지원 입력 파라미터 (Input Attributes):
- `location`: 대상 지점 위경도 좌표 (서울, 제주도, 독도, 뉴욕)
- `timestamp`: UTC 기준 타임스탬프 (한국의 서머타임 미적용 여부 및 해외 서머타임 정확 계산)
- `language`: `"ko"`

### 📤 반환 출력 속성 (Output Attributes):
- `timeZoneId`: `"Asia/Seoul"`, `"America/New_York"` 등
- `timeZoneName`: `"한국 표준시"`, `"동부 하계 표준시"` 등
- `rawOffset`: 표준 UTC 오프셋 (한국: +32,400초 = +9시간)
- `dstOffset`: 서머타임(DST) 오프셋 (한국: 0시간)
- `status`: `"OK"`

### 💡 실무 적용 시나리오:
1. **국내외 출장/항공/호텔 예약 시스템 현지 시간 계산**: 인천/김포공항 출발 ➡️ 해외 도착지 시간대 및 시차 자동 계산
2. **국내 IoT 스마트 기기 시계 동기화**: 서울/제주/독도 좌표 기반 한국 표준시(KST, UTC+9) 자동 설정


In [ ]:
seoul_compare_cities = [
    {"city": "대한민국 서울 (KST)", "coords": (37.5665, 126.9780)},
    {"city": "대한민국 제주도", "coords": (33.3617, 126.5332)},
    {"city": "대한민국 독도", "coords": (37.2427, 131.8683)},
    {"city": "미국 뉴욕 (서머타임 비교)", "coords": (40.7128, -74.0060)}
]

now_timestamp = datetime.datetime.now(datetime.timezone.utc).timestamp()

try:
    timezone_records = []
    for c in seoul_compare_cities:
        tz_res = gmaps.timezone(location=c["coords"], timestamp=now_timestamp, language="ko")
        if tz_res.get("status") == "OK":
            raw_h = tz_res.get("rawOffset", 0) / 3600
            dst_h = tz_res.get("dstOffset", 0) / 3600
            total_h = raw_h + dst_h
            timezone_records.append({
                "도시/지역명": c["city"],
                "타임존 ID": tz_res.get("timeZoneId"),
                "타임존 명칭": tz_res.get("timeZoneName"),
                "표준 오프셋": f"{raw_h:+.1f} 시간",
                "서머타임 (DST)": f"{dst_h:+.1f} 시간" if dst_h != 0 else "미적용 (0h)",
                "최종 UTC 오프셋": f"UTC{total_h:+.1f}"
            })
    df_tz = pd.DataFrame(timezone_records)
    print("✅ 서울 및 주요 지역 타임존/서머타임 조회 결과:")
    display(df_tz)
except Exception as e:
    print("⚠️ Time Zone API 참고:", e)
    print("💡 콘솔 활성화 링크: https://console.cloud.google.com/apis/library/timezone-backend.googleapis.com")


## 📶 11. Geolocation API (서울 시내 네트워크/기지국/IP 기반 위치 추정)

### 📥 지원 입력 파라미터 (Input Attributes):
- `considerIp`: 한국 IP 주소 기반 위치 추정 포함 여부 (`true`/`false`)
- `radioType`: 무선 통신 규격 (`"lte"`, `"nr"` 5G, `"wcdma"`)
- `cellTowers` / `wifiAccessPoints`: 서울 시내 기지국 신호 및 주변 Wi-Fi AP BSSID 정보

### 📤 반환 출력 속성 (Output Attributes):
- `location`: 추정 위도(`lat`), 경도(`lng`)
- `accuracy`: 위치 정확도 오차 반경 (미터 단위)

### 💡 실무 적용 시나리오:
1. **서울 지하철 및 대형 지하 쇼핑몰(코엑스몰 등) 실내 측위**: GPS 신호 수신이 불가능한 지하 공간에서 주변 Wi-Fi 신호로 현재 위치 계산
2. **서울 저전력 IoT 물류 트래킹**: LTE 기지국 및 Wi-Fi AP 스캔만으로 도심 배송 화물 위치 추적


In [ ]:
try:
    geolocate_res = gmaps.geolocate(consider_ip=True)
    print("✅ Geolocation 기기 위치 추정 성공:")
    print(f"  - 추정 좌표: {geolocate_res.get('location')}")
    print(f"  - 정확도 반경: {geolocate_res.get('accuracy')} 미터")
    print_json(geolocate_res, title="Geolocation API 응답 JSON")
except Exception as e:
    print("⚠️ Geolocation API 참고:", e)
    print("💡 콘솔 활성화 링크: https://console.cloud.google.com/apis/library/geolocation.googleapis.com")


## 🛣️ 12. Roads API (서울 강남 테헤란로 도로 스냅 Snap to Roads)

### 📥 지원 입력 파라미터 (Input Attributes):
- `path`: 서울 강남 테헤란로를 따라 주행 중 수집된 약간의 GPS 오차가 있는 좌표 리스트
- `interpolate`: 빌딩 숲으로 인한 음영 구간을 실제 도로 형상에 맞춰 자동 보간 (`True`)

### 📤 반환 출력 속성 (Output Attributes):
- `snappedPoints`:
  - `location`: 테헤란로 도로 중심선에 정확히 스냅된 보정 위도/경도
  - `originalIndex`: 원본 수집 좌표 매핑 인덱스
  - `placeId`: 테헤란로 도로 세그먼트의 고유 Place ID

### 💡 실무 적용 시나리오:
1. **서울 강남 빌딩 숲 GPS 궤적 튐 보정**: 고층 빌딩 반사로 인해 인도나 건물 내부로 튀는 차량 GPS 궤적을 테헤란로 도로 위로 정밀 보정
2. **서울 법정 속도 제한(5030 안전속도) 준수율 관제**: 스냅된 도로 세그먼트의 제한속도와 실시간 속도를 대조하여 안전운전 지수 산정


In [ ]:
# 서울 강남구 테헤란로(강남역 -> 역삼역 -> 선릉역) 주행 중 수집된 GPS 궤적 예시
seoul_teheran_path = [
    (37.4981, 127.0276),  # 강남역 부근
    (37.5000, 127.0365),  # 역삼역 (GFC) 부근
    (37.5015, 127.0430),  # 르네상스호텔 사거리
    (37.5045, 127.0490)   # 선릉역 부근
]

try:
    snapped = gmaps.snap_to_roads(seoul_teheran_path, interpolate=True)
    print(f"✅ 서울 테헤란로 도로 스냅 완료 (원본 {len(seoul_teheran_path)}개 ➡️ 보정 {len(snapped)}개 지점):\n")
    
    snap_records = []
    for idx, pt in enumerate(snapped):
        snap_records.append({
            "포인트": f"Snap #{idx + 1}",
            "보정 위도": pt.get("location", {}).get("latitude"),
            "보정 경도": pt.get("location", {}).get("longitude"),
            "원본 매핑 인덱스": pt.get("originalIndex", "보간 지점(Interpolated)"),
            "도로 Place ID": pt.get("placeId")
        })
    df_snapped = pd.DataFrame(snap_records)
    display(df_snapped)
except Exception as e:
    print("⚠️ Roads API 참고:", e)
    print("💡 콘솔 활성화 링크: https://console.cloud.google.com/apis/library/roads.googleapis.com")


## 📊 13. 서울 중심 속성 매트릭스 & 실무 활용 레퍼런스

| API 서비스 | 주요 입력 파라미터 (Inputs) | 핵심 출력 속성 (Outputs) | 서울 중심 대표 실무 활용 시나리오 |
| :--- | :--- | :--- | :--- |
| **Geocoding** | `address`, `components`, `bounds`, `region`, `language` | `formatted_address`, `geometry(location, type, viewport)`, `address_components`, `place_id` | 강남파이낸스센터(GFC) 주소 표준화, 우편번호(06236) 검증, 서울 지도 뷰포트 피팅 |
| **Reverse Geocoding** | `latlng`, `result_type`, `location_type`, `language` | 계층형 주소 목록 (`street_address` ~ `country`), `types`, `place_id` | 강남/역삼 GPS 좌표를 도로명 주소로 변환, 배달/택시 픽업 지점 자동 설정 |
| **Places (New) Search** | `textQuery`, `includedTypes`, `locationRestriction`, `openNow`, `minRating`, `rankPreference` | 장소 목록, `displayName`, `rating`, `userRatingCount`, `regularOpeningHours`, `location` | 강남역/성수동 "영업중 + 평점 4.0+" 카페/맛집 탐색, 반경 1.5km 시설 거리순 정렬 |
| **Places (New) Details** | `placeId`, `X-Goog-FieldMask: *`, `languageCode` | 50+ 속성 (`reviews`, `photos`, `editorialSummary`, `accessibility`, `parking`, `evCharge`, `dineIn`) | 코엑스/더현대서울 매장 상세 프로필 구축, 무장애(휠체어)/EV 충전 시설 필터링 |
| **Places Autocomplete** | `input`, `sessionToken`, `origin`, `includedRegionCodes: ["kr"]` | `suggestions(mainText, secondaryText, distanceMeters, placeId)` | 서울시청 기준 "남산", "코엑스" 실시간 자동완성, 거리(km) 표기, 세션 토큰 과금 절감 |
| **Directions** | `origin`, `destination`, `mode`, `waypoints(optimize=True)`, `traffic_model`, `avoid` | `legs(distance, duration, traffic)`, `steps(turn-by-turn)`, `waypoint_order`, `polyline` | 서울역 ➡️ 명동, 남산, 잠실 다중 경유지 최적 순서 배송(TSP), 올림픽대로 교통 반영 ETA |
| **Distance Matrix** | `origins[]`, `destinations[]`, `mode`, `departure_time`, `traffic_model` | N x M 요소 행렬 (`distance`, `duration`, `duration_in_traffic`, `status`) | 3개 출발역(서울역, 용산역, 청량리역) ➡️ 2개 목적지(강남역, 여의도) 간 최적 배차 매트릭스 |
| **Elevation** | `locations[]`, `path[]`, `samples` | `elevation`(해발 고도 m), `resolution`, 고도 프로파일 | 북한산, 남산타워, 롯데월드타워 고도 비교, 서울역 ➡️ 남산타워 오르막 경로 고도 분석 |
| **Time Zone** | `location`, `timestamp`, `language` | `timeZoneId`, `timeZoneName`, `rawOffset`, `dstOffset` (서머타임) | 대한민국 서울(KST, UTC+9) 표준시 확인 및 글로벌 예약 시차 계산 |
| **Geolocation** | `considerIp`, `radioType`, `cellTowers[]`, `wifiAccessPoints[]` | `location(lat, lng)`, `accuracy`(오차 반경 m) | 서울 지하철/코엑스몰 지하 실내 측위, 저전력 IoT 화물 위치 추적 |
| **Roads** | `path[]`, `interpolate`, `points[]` | `snappedPoints(location, originalIndex, placeId)` | 강남 테헤란로 빌딩 숲 GPS 궤적 도로 스냅 보정, 서울 5030 안전속도 준수율 관제 |

---

### 📚 공식 개발자 문서 & 레퍼런스
- 🌐 [Google Maps Platform 공식 문서](https://developers.google.com/maps/documentation)
- 🏢 [Places API (New) Web Service 개요](https://developers.google.com/maps/documentation/places/web-service/op-overview)
- 🚗 [Directions API 개발자 가이드](https://developers.google.com/maps/documentation/directions/overview)
- 🐍 [google-maps-services-python GitHub](https://github.com/googlemaps/google-maps-services-python)
- ⚙️ [Google Cloud Console API 라이브러리](https://console.cloud.google.com/apis/library)
